# R&R Clustering Visuals

This notebook reads compact R&R artifact datasets and regenerates clustering plus topic/style figures from them.


In [ ]:
from pathlib import Path
import textwrap

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import Normalize
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.manifold import TSNE

sns.set_theme(style="white", context="paper")
mpl.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "font.family": "DejaVu Sans",
    "axes.edgecolor": "#1B1F24",
    "axes.labelcolor": "#1E2026",
    "xtick.color": "#2E3138",
    "ytick.color": "#2E3138",
    "axes.titleweight": "bold",
    "savefig.facecolor": "white",
    "savefig.edgecolor": "white",
})

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "rr").exists() and (PROJECT_ROOT.parent / "rr").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

ARTIFACT_DIR = PROJECT_ROOT / "rr" / "artifacts"
CLUSTER_PATH = ARTIFACT_DIR / "cluster_assignments_with_original_idx.parquet"
TOPIC_RATING_PATH = ARTIFACT_DIR / "topic_rating_table.csv"
STYLE_RATING_PATH = ARTIFACT_DIR / "style_rating_table.csv"
INTERACTION_PATH = ARTIFACT_DIR / "topic_style_interaction_table.csv"
FIG_DIR = PROJECT_ROOT / "results" / "figures" / "clustering_visuals"
FIG_DIR.mkdir(parents=True, exist_ok=True)

RATING_MIN, RATING_MAX = 1.0, 5.0
PALETTE = "RdYlBu_r"
RANDOM_STATE = 42

print(f"Reading clustering data from: {CLUSTER_PATH}")
print(f"Saving figures to: {FIG_DIR}")


In [ ]:
cluster_assignments = pd.read_parquet(CLUSTER_PATH)
topic_rating_table = pd.read_csv(TOPIC_RATING_PATH)
style_rating_table = pd.read_csv(STYLE_RATING_PATH)
interaction_table = pd.read_csv(INTERACTION_PATH)
cluster_assignments["rating"] = pd.to_numeric(cluster_assignments["rating"], errors="coerce")
cluster_assignments["cluster"] = pd.to_numeric(cluster_assignments["cluster"], errors="coerce").astype("Int64")
cluster_assignments["reasoning_clean"] = cluster_assignments["reasoning_clean"].fillna("").astype(str)

global_avg_rating = float(cluster_assignments["rating"].mean())
noise_share = float((cluster_assignments["cluster"] == -1).mean())

print("Cluster assignments:", cluster_assignments.shape)
print("Topic table:", topic_rating_table.shape)
print("Style table:", style_rating_table.shape)
print("Interaction table:", interaction_table.shape)
print(f"Global average rating: {global_avg_rating:.3f}")
print(f"Noise share: {noise_share:.1%}")
print("Rating counts:")
print(cluster_assignments["rating"].value_counts().sort_index())


In [ ]:
def wrap_label(value, width=34):
    return textwrap.fill(str(value), width=width)


def save_current(fig, stem):
    for ext in ["png", "pdf", "svg"]:
        fig.savefig(FIG_DIR / f"{stem}.{ext}", dpi=300, bbox_inches="tight")
    print(f"Saved {stem} to {FIG_DIR}")


def safe_label(df, id_col, label_col):
    if label_col in df.columns:
        return df[label_col].fillna(df[id_col]).astype(str)
    return df[id_col].astype(str)


def plot_fixed_theme_lollipop(table, id_col, label_col, title, output_stem, max_rows, empty_message):
    if len(table) == 0:
        print(empty_message)
        return

    plot_data = table.copy()
    plot_data["display_label"] = safe_label(plot_data, id_col, label_col)
    plot_data = plot_data.sort_values(["avg_rating", "support"], ascending=[True, False])
    plot_data = plot_data.tail(min(max_rows, len(plot_data))).reset_index(drop=True)

    y = np.arange(len(plot_data))
    point_size = 150
    rating_min, rating_max = RATING_MIN, RATING_MAX

    econ_black = "#1B1F24"
    econ_grey = "#4A4F57"
    econ_light = "#D6D9DE"
    econ_lighter = "#EEF1F4"
    econ_bg = "#FFFFFF"
    lollipop_color = "#5E7EA6"

    rc = {
        "figure.facecolor": econ_bg,
        "axes.facecolor": econ_bg,
        "font.family": "DejaVu Sans",
        "font.size": 11.5,
        "axes.titlesize": 15.5,
        "axes.titleweight": "bold",
        "axes.labelsize": 12.5,
        "axes.edgecolor": econ_black,
        "axes.linewidth": 0.9,
        "xtick.color": econ_grey,
        "ytick.color": econ_grey,
        "xtick.labelsize": 11,
        "ytick.labelsize": 11,
        "grid.color": econ_light,
        "grid.linewidth": 0.8,
        "grid.alpha": 1.0,
        "legend.frameon": False,
    }

    with plt.rc_context(rc):
        fig, ax = plt.subplots(figsize=(13, max(6, 0.42 * len(plot_data) + 2.0)), dpi=180)
        ax.set_axisbelow(True)

        for yi, x in zip(y, plot_data["avg_rating"]):
            ax.hlines(
                yi,
                global_avg_rating,
                x,
                color=lollipop_color,
                linewidth=3.2,
                alpha=0.9,
                zorder=2,
                capstyle="round",
            )

        ax.scatter(
            plot_data["avg_rating"],
            y,
            s=point_size,
            color=lollipop_color,
            edgecolor=econ_black,
            linewidth=0.7,
            zorder=3,
        )

        n_col_x = rating_max + 0.08
        for yi, n in zip(y, plot_data["support"]):
            ax.text(
                n_col_x,
                yi,
                f"n={int(n):,}",
                va="center",
                ha="left",
                fontsize=10,
                color=econ_grey,
                zorder=5,
                bbox={"facecolor": econ_bg, "edgecolor": "none", "pad": 0.15, "alpha": 0.9},
            )

        ax.axvline(
            global_avg_rating,
            color=econ_black,
            linestyle=(0, (4, 3)),
            linewidth=1.3,
            alpha=0.95,
            zorder=2,
        )

        ax.set_yticks(y)
        ax.set_yticklabels([wrap_label(x, 34) for x in plot_data["display_label"]])
        ax.set_xlim(rating_min - 0.02, rating_max + 0.34)
        ax.set_xlabel("Average moral rating")
        ax.set_ylabel("")
        ax.set_title(title, pad=14, loc="left")

        ax.grid(axis="x", which="major", color=econ_light, linewidth=0.8)
        ax.grid(axis="x", which="minor", color=econ_lighter, linewidth=0.7)
        ax.grid(axis="y", visible=False)
        ax.minorticks_on()
        ax.tick_params(axis="y", length=0)
        ax.tick_params(axis="x", which="major", length=4, width=0.9, color=econ_grey)
        ax.tick_params(axis="x", which="minor", length=2, width=0.7, color=econ_light)

        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.spines["left"].set_visible(False)
        ax.spines["bottom"].set_color(econ_black)

        fig.tight_layout()
        save_current(fig, output_stem)
        plt.show()


def top_terms_by_cluster(df, top_k=4):
    non_noise = df[df["cluster"] != -1].copy()
    vectorizer = TfidfVectorizer(
        ngram_range=(1, 3),
        min_df=8,
        max_df=0.75,
        stop_words="english",
    )
    tfidf = vectorizer.fit_transform(non_noise["reasoning_clean"])
    terms = np.array(vectorizer.get_feature_names_out())
    labels = non_noise["cluster"].to_numpy()

    rows = []
    for cluster, size in non_noise["cluster"].value_counts().items():
        mask = labels == cluster
        group_mean = np.asarray(tfidf[mask].mean(axis=0)).ravel()
        rest_mean = np.asarray(tfidf[~mask].mean(axis=0)).ravel() if (~mask).sum() else np.zeros_like(group_mean)
        lift = group_mean - rest_mean
        top_idx = np.argsort(lift)[::-1][:top_k]
        rows.append({
            "cluster": int(cluster),
            "size": int(size),
            "avg_rating": float(non_noise.loc[mask, "rating"].mean()),
            "top_terms": ", ".join(terms[i] for i in top_idx),
        })
    return pd.DataFrame(rows).sort_values("size", ascending=False).reset_index(drop=True)


cluster_summary = top_terms_by_cluster(cluster_assignments)
cluster_summary.head()


## Cluster Rating Plot


In [ ]:
plot_df = cluster_summary.head(28).copy()
plot_df["label"] = plot_df.apply(lambda r: f"C{int(r['cluster'])}: {r['top_terms']}", axis=1)
plot_df = plot_df.sort_values("avg_rating", ascending=True).reset_index(drop=True)

y = np.arange(len(plot_df))
norm = Normalize(vmin=RATING_MIN, vmax=RATING_MAX)
colors = plt.cm.get_cmap(PALETTE)(norm(plot_df["avg_rating"].to_numpy()))
sizes = 60 + 12 * np.sqrt(plot_df["size"].to_numpy(dtype=float))

fig, ax = plt.subplots(figsize=(13.5, max(7, 0.36 * len(plot_df) + 2.2)), dpi=180)
for yi, x, c in zip(y, plot_df["avg_rating"], colors):
    ax.hlines(yi, global_avg_rating, x, color=c, linewidth=3.0, alpha=0.9, capstyle="round")
ax.scatter(plot_df["avg_rating"], y, s=sizes, c=plot_df["avg_rating"], cmap=PALETTE, norm=norm, edgecolor="#1B1F24", linewidth=0.7)

for yi, x, n in zip(y, plot_df["avg_rating"], plot_df["size"]):
    ha = "left" if x >= global_avg_rating else "right"
    tx = min(float(x) + 0.05, RATING_MAX + 0.05) if ha == "left" else max(float(x) - 0.05, RATING_MIN - 0.05)
    ax.text(tx, yi, f"n={int(n):,}", va="center", ha=ha, fontsize=8.8, color="#4A4F57")

ax.axvline(global_avg_rating, color="#1B1F24", linestyle=(0, (4, 3)), linewidth=1.2)
ax.set_yticks(y)
ax.set_yticklabels([wrap_label(x, 42) for x in plot_df["label"]], fontsize=9.4)
ax.set_xlim(RATING_MIN - 0.08, RATING_MAX + 0.18)
ax.set_xlabel("Average moral rating")
ax.set_ylabel("")
ax.set_title("Largest Reasoning Clusters by Average Moral Rating", loc="left", pad=12, fontsize=14)
ax.grid(axis="x", color="#D6D9DE", linewidth=0.8)
ax.grid(axis="y", visible=False)
ax.tick_params(axis="y", length=0)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)

cbar = fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=PALETTE), ax=ax, pad=0.01)
cbar.set_label("Average rating", rotation=90)
cbar.outline.set_visible(False)
cbar.ax.tick_params(length=0, labelsize=8.5)

fig.tight_layout()
save_current(fig, "cluster_rating_lollipop")
plt.show()


## Rating t-SNE


In [ ]:
def sample_per_group(df, group_col, max_per_group, random_state=42):
    parts = []
    for _, group in df.groupby(group_col, sort=True):
        n_take = min(max_per_group, len(group))
        parts.append(group.sample(n=n_take, random_state=random_state))
    return pd.concat(parts, ignore_index=True) if parts else df.iloc[0:0].copy()


def build_text_embedding(text_series, max_features=6000, svd_components=60, random_state=42):
    vectorizer = TfidfVectorizer(
        stop_words="english",
        max_features=max_features,
        ngram_range=(1, 2),
        min_df=2,
    )
    sparse = vectorizer.fit_transform(text_series.fillna("").astype(str))
    n_comp = min(svd_components, sparse.shape[0] - 1, sparse.shape[1] - 1)
    if n_comp >= 2:
        return TruncatedSVD(n_components=n_comp, random_state=random_state).fit_transform(sparse)
    return sparse.toarray()


def run_tsne(matrix, random_state=42):
    n = matrix.shape[0]
    perplexity = min(35, max(5, (n - 1) // 4))
    return TSNE(
        n_components=2,
        init="pca",
        learning_rate="auto",
        perplexity=perplexity,
        random_state=random_state,
    ).fit_transform(matrix)


rating_levels = [1, 2, 3, 4, 5]
tsne_df = cluster_assignments[cluster_assignments["rating"].isin(rating_levels)][["reasoning_clean", "rating", "cluster"]].copy()
tsne_df = sample_per_group(tsne_df, "rating", max_per_group=900, random_state=RANDOM_STATE)
xy = run_tsne(build_text_embedding(tsne_df["reasoning_clean"], random_state=RANDOM_STATE), random_state=RANDOM_STATE)
tsne_df["tsne_x"] = xy[:, 0]
tsne_df["tsne_y"] = xy[:, 1]
tsne_df["rating"] = tsne_df["rating"].astype(int)

rating_palette = {
    1: "#355070",
    2: "#6d597a",
    3: "#b56576",
    4: "#e56b6f",
    5: "#eaac8b",
}

fig, ax = plt.subplots(figsize=(13, 9), dpi=160)
sns.scatterplot(
    data=tsne_df,
    x="tsne_x",
    y="tsne_y",
    hue="rating",
    hue_order=rating_levels,
    palette=rating_palette,
    s=24,
    alpha=0.8,
    linewidth=0,
    ax=ax,
)

for rating in rating_levels:
    subset = tsne_df[tsne_df["rating"] == rating]
    if len(subset) == 0:
        continue
    ax.text(
        subset["tsne_x"].mean(),
        subset["tsne_y"].mean(),
        f"R{rating}",
        fontsize=11,
        weight="bold",
        ha="center",
        va="center",
        bbox={"boxstyle": "round,pad=0.22", "fc": "white", "ec": rating_palette[rating], "lw": 1.4, "alpha": 0.92},
    )

ax.set_title("t-SNE of Reasoning Traces by Moral Rating", fontsize=19, pad=14)
ax.set_xlabel("t-SNE dimension 1")
ax.set_ylabel("t-SNE dimension 2")
ax.grid(True)
ax.legend(title="Moral rating", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=True)
fig.tight_layout()
save_current(fig, "rating_tsne_panel")
plt.show()


## Topic And Style Visuals


In [ ]:
if len(interaction_table) == 0:
    print("Interaction table is empty; skipping.")
else:
    top_topics = interaction_table.groupby("topic")["support"].sum().sort_values(ascending=False).head(9).index.tolist()
    top_styles = interaction_table.groupby("style")["support"].sum().sort_values(ascending=False).head(9).index.tolist()
    subset = interaction_table[interaction_table["topic"].isin(top_topics) & interaction_table["style"].isin(top_styles)].copy()

    topic_map = dict(zip(topic_rating_table["topic"], topic_rating_table.get("topic_label", topic_rating_table["topic"])))
    style_map = dict(zip(style_rating_table["style"], style_rating_table.get("style_label", style_rating_table["style"])))

    avg_mat = subset.pivot_table(index="topic", columns="style", values="avg_rating", aggfunc="mean").reindex(index=top_topics, columns=top_styles)
    sup_mat = subset.pivot_table(index="topic", columns="style", values="support", aggfunc="sum").reindex(index=top_topics, columns=top_styles)

    avg_plot = avg_mat.copy()
    avg_plot.index = [wrap_label(topic_map.get(t, t), 28) for t in avg_plot.index]
    avg_plot.columns = [wrap_label(style_map.get(s, s), 24) for s in avg_plot.columns]

    annot = avg_mat.copy().astype(object)
    for i in range(annot.shape[0]):
        for j in range(annot.shape[1]):
            v = avg_mat.iat[i, j]
            n = sup_mat.iat[i, j]
            annot.iat[i, j] = "" if pd.isna(v) else f"{v:.2f}\n(n={int(n):,})"

    fig, ax = plt.subplots(figsize=(15, 9), dpi=180)
    sns.heatmap(
        avg_plot, mask=avg_plot.isna(),
        cmap=sns.diverging_palette(240, 10, s=85, l=55, as_cmap=True),
        vmin=RATING_MIN, vmax=RATING_MAX,
        linewidths=0.55, linecolor="#E6E0D5",
        annot=annot.values, fmt="",
        annot_kws={"fontsize": 8.0, "color": "#1E2026", "va": "center"},
        cbar_kws={"label": "Average moral rating", "shrink": 0.90, "pad": 0.02, "aspect": 30},
        ax=ax,
    )
    ax.set_title("High-Support Topic x Style Interactions", loc="left", pad=10, fontsize=14)
    ax.set_xlabel("Reasoning style")
    ax.set_ylabel("Topic framing")
    ax.tick_params(axis="x", labelrotation=35, length=0, pad=2)
    ax.tick_params(axis="y", labelrotation=0, length=0, pad=2)
    fig.tight_layout()
    save_current(fig, "topic_style_heatmap")
    plt.show()


In [ ]:
plot_fixed_theme_lollipop(
    topic_rating_table,
    id_col="topic",
    label_col="topic_label",
    title="Topic Framing and Moral Rating",
    output_stem="topic_rating_lollipop_fixed_theme",
    max_rows=24,
    empty_message="Topic table is empty; skipping topic lollipop chart.",
)


In [ ]:
plot_fixed_theme_lollipop(
    style_rating_table,
    id_col="style",
    label_col="style_label",
    title="Reasoning Style and Moral Rating",
    output_stem="style_rating_lollipop_fixed_theme",
    max_rows=20,
    empty_message="Style table is empty; skipping style lollipop chart.",
)
